In [1]:
pip install pinecone

Note: you may need to restart the kernel to use updated packages.


In [30]:
from pinecone import Pinecone, ServerlessSpec
from dotenv import load_dotenv
import os
# Load environment variables
load_dotenv()
# Get API key for Euri
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
EURI_API_KEY = os.getenv("EURI_API_KEY")

pc = Pinecone(api_key=PINECONE_API_KEY)
print(pc)

In [13]:
#Create Index
index_name = "mytestindex"

pc.create_index(
    name=index_name,
    dimension=1536, # Replace with your embedding model dimensions
    metric="cosine", # Replace with your model metric
    spec=ServerlessSpec(
        cloud="aws",
        region="us-east-1"
    ) 
)

{
    "name": "mytestindex",
    "metric": "cosine",
    "host": "mytestindex-ukt67zh.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 1536,
    "deletion_protection": "disabled",
    "tags": null
}

In [12]:
#Delete Index
pc.delete_index("mytestindex")

In [10]:
#Create Again
index_name = "mytestindex"

pc.create_index(
    name=index_name,
    dimension=1536, # Replace with your embedding model dimensions
    metric="cosine", # Replace with your model metric
    spec=ServerlessSpec(
        cloud="aws",
        region="us-east-1"
    ) 
)

{
    "name": "mytestindex",
    "metric": "cosine",
    "host": "mytestindex-ukt67zh.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 1536,
    "deletion_protection": "disabled",
    "tags": null
}

In [14]:
pc.list_indexes()

[
    {
        "name": "medprep",
        "metric": "cosine",
        "host": "medprep-ukt67zh.svc.aped-4627-b74a.pinecone.io",
        "spec": {
            "serverless": {
                "cloud": "aws",
                "region": "us-east-1"
            }
        },
        "status": {
            "ready": true,
            "state": "Ready"
        },
        "vector_type": "dense",
        "dimension": 384,
        "deletion_protection": "disabled",
        "tags": null
    },
    {
        "name": "mytestindex",
        "metric": "cosine",
        "host": "mytestindex-ukt67zh.svc.aped-4627-b74a.pinecone.io",
        "spec": {
            "serverless": {
                "cloud": "aws",
                "region": "us-east-1"
            }
        },
        "status": {
            "ready": true,
            "state": "Ready"
        },
        "vector_type": "dense",
        "dimension": 1536,
        "deletion_protection": "disabled",
        "tags": null
    },
    {
        "name":

In [42]:
import requests
import numpy as np

def generate_embeddings(data):
    url = "https://api.euron.one/api/v1/euri/alpha/embeddings"
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {EURI_API_KEY}"
    }
    payload = {
        "input":data,
        "model": "text-embedding-3-small"
    }

    response = requests.post(url, headers=headers, json=payload)
    data = response.json()
    
    # Convert to numpy array for vector operations
    embedding = np.array(data['data'][0]['embedding'])
    
    print(f"Generated embedding with shape: {embedding.shape}")
    print(f"First 5 values: {embedding[:5]}")
    
    # Example: Calculate vector norm
    norm = np.linalg.norm(embedding)
    print(f"Vector norm: {norm}")
    
    return embedding

In [43]:
text1 = "The patient is diagnosed with Opioid Use Disorder today"
text2 = "He is planning to start medication"
text3 = "Dr. Nalin prescribed Buprenorphine to this patient"
texts = [text1, text2, text3]

text = ". ".join(texts)

In [40]:
text

'The patient is diagnosed with Opioid Use Disorder today. He is planning to start medication. Dr. Nalin prescribed Buprenorphine to this patient'

In [45]:
embeddings_data = generate_embeddings(text)

Generated embedding with shape: (1536,)
First 5 values: [-0.03119289  0.02218569 -0.01259404  0.04803843 -0.02445468]
Vector norm: 0.9999999871637589


In [46]:
index = pc.Index('mytestindex')
index.upsert([("item-id-001",embeddings_data.tolist(),{"name":"First Response"})])

{'upserted_count': 1}

In [47]:

index.upsert( vectors=[ { "id": "item-id-002", "values": embeddings_data.tolist(), "metadata": { "text": "Lorem Ipsum is simply dummy text of the printing and typesetting industry." } } ], batch_size=1 )

Upserted vectors: 100%|██████████| 1/1 [00:01<00:00,  1.06s/it]


{'upserted_count': 1}

In [48]:
index.upsert( vectors =[ ("item-id-003", embeddings_data.tolist(),{"name":"test", "info":"personal info"}) ] )

{'upserted_count': 1}

In [49]:
index.fetch(ids=["item-id-002"])

FetchResponse(namespace='', vectors={'item-id-002': Vector(id='item-id-002', values=[-0.0311928932, 0.022185687, -0.0125940442, 0.0480384305, -0.0244546775, 0.0163069386, 0.0180717092, 0.0501469858, 0.00901293568, -0.0270903744, 0.00205985387, -0.00450933259, -0.0367393158, 0.0115454961, 0.0563809797, -0.00852017477, 0.00419419492, -0.00611366937, -0.046984151, 0.0154016335, 0.0356850363, 0.00800449494, 0.0242942441, -0.0375644, -0.00896136742, -0.0306428336, 0.0128919929, -0.0206959452, 0.0287634674, -0.00405668048, -0.0278237853, -0.0143588148, 0.00728827342, -0.0348828658, -0.00332613406, -0.0186790656, 0.025761066, 0.0316054374, 0.000493476808, 0.00524560874, -0.0489551947, -0.100935705, 0.0105084069, 0.00649756426, -0.0185071714, -0.000996980816, -0.00917336904, -0.0327055529, 0.0326138772, -0.0309407823, -0.0409793481, -0.0545016155, -0.0505595319, 0.00610221, -0.0161808841, -0.00191374472, 0.023148289, 0.0246380307, -0.00524560874, -0.00572404452, 0.0219564959, 0.00173182436, 0.

In [ ]:
text = "my name is sudhanshu"
embedings_to_search = generate_embeddings(text)
result = index.query(vector=embedings_to_search.tolist(),
            top_k = 2,
            include_metadata = True)
print(result)

In [50]:
text = "What is the doctors name?"
embedings_to_search = generate_embeddings(text)
result = index.query(vector=embedings_to_search.tolist(),
            top_k = 2,
            include_metadata = True)
print(result)

Generated embedding with shape: (1536,)
First 5 values: [ 0.02148603  0.01613097  0.0017927   0.03784068 -0.04194578]
Vector norm: 1.0000000382631116
{'matches': [{'id': 'item-id-002',
              'metadata': {'text': 'Lorem Ipsum is simply dummy text of the '
                                   'printing and typesetting industry.'},
              'score': 0.281230927,
              'values': []},
             {'id': 'item-id-001',
              'metadata': {'name': 'First Response'},
              'score': 0.281230927,
              'values': []}],
 'namespace': '',
 'usage': {'read_units': 6}}
